# Complete the full SRS report

Notebook 08 fills in `requirement_reviews` only. This notebook adds the other three sections so
`data/report.json` qualifies as a `FULL_SRS` run (Task #8, issue #33):

| Section | What it covers | Benchmark cases |
|---|---|---|
| `relationships` | The three benchmark requirement pairs | REL-001 to REL-003 |
| `document_findings` | Document-wide checks, including the unnumbered §2.1 "shall" statements | DOC-001 to DOC-005 |
| `swe_coverage` | One evidence summary per NPR §4.1 SWE | SWE-001 to SWE-006 |

**How it works.** One Gemini call per section (3 calls total, to stay under the free tier's daily
limit). Like 08, the LLM only makes the decisions and cites NASA clauses by ID; the code fills in the
case IDs, requirement IDs, and exact NASA sections and URLs. Every answer is checked with the models in
`report_schema.py`, and any failure is sent back to the LLM to fix.

`requirement_reviews` is left unchanged.

**Setup.** Same as 08: `GOOGLE_API_KEY=...` in `.env` at the repo root. Answers are cached in
`data/llm_cache/`, so re-running doesn't call the API again. Delete a cache file to redo that section.

In [1]:
import json, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "rag_retrievals.csv").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "contract"))
from report_schema import (CORE_SWE_SECTIONS, NPR_CHAPTER4_URL, DocumentCriterion, DocumentEvidence,
                           DocumentFinding, DocumentStatus, Relationship, RelationshipIssueCode,
                           RelationshipStatus, Report, SweCoverage)
from review_tools import Quote, clauses, locator, nasa_ref, srs, srs_text, with_api_retries
from validate_report import validate

load_dotenv(ROOT / ".env")

MODEL = "gemini-3.5-flash"  # same model as 08
PROMPT_VERSION = 1          # bump when a prompt below changes, so old cached answers aren't reused
CACHE_DIR = ROOT / "data" / "llm_cache" / f"{MODEL}_full_report_v{PROMPT_VERSION}"
REPORT_PATH = ROOT / "data" / "report.json"
MAX_ATTEMPTS = 3
PAUSE_SECONDS = 4

In [2]:
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
benchmark = pd.read_csv(ROOT / "data" / "benchmark" / "benchmark_inputs_for_teams_v1.csv")
calibration = pd.read_csv(ROOT / "data" / "benchmark" / "calibration_examples_for_teams_v1.csv")

rel_cases = benchmark[benchmark.scope == "RELATIONSHIP"]
doc_cases = benchmark[benchmark.scope == "DOCUMENT"]
swe_cases = benchmark[benchmark.scope == "SWE_COVERAGE"]

print(len(report["requirement_reviews"]), "requirement reviews already in report.json")
print(len(rel_cases), "relationship cases |", len(doc_cases), "document cases |", len(swe_cases), "SWE cases")

54 requirement reviews already in report.json
3 relationship cases | 5 document cases | 6 SWE cases


## What the LLM sees

- The whole SRS (about 12 KB), so it can check any section.
- A short preview of each of the 62 NASA clauses, cited by the ID in square brackets.
- For SWE coverage, the full text of each §4.1 provision.

The rules below come from the output contract (`contract/output_contract_v1.md`) and the issue. They
are written in general terms only: no benchmark answer is in any prompt, so the calibration comparison
at the end stays a fair check.

In [3]:
clause_menu = "\n".join(f"[{cid}] {row.source} §{row.section}: {row.text[:300]}" for cid, row in clauses.iterrows())

# Text of each §4.1 provision, from the full clause extraction (some aren't in the 62 addressable clauses).
all_clauses = pd.DataFrame(json.loads((ROOT / "data" / "clauses.json").read_text(encoding="utf-8")))
swe_text = {row.swe_id: row.text for row in all_clauses.itertuples()
            if row.type == "requirement" and row.swe_id in CORE_SWE_SECTIONS}

RULES = '''You are auditing the Starhawk Mission Computer SRS (SH-SRS-001, the SFMC) against NASA NPR 7150.2D
and the NASA software engineering handbook topic 5.09 (NASA-HDBK-2203).

General rules
- Judge only what is visible in the SRS. The SRS alone can never prove that a NASA project complied with the NPR.
- Handbook text is guidance. A missing Handbook-recommended section is a guidance gap, not an NPR violation.
- Every quote must be copied exactly from the SRS and kept short. For a quote from a numbered requirement, use the
  locator given for it (e.g. "§3.3, SFMC-REQ-015"). For other text, give its section (e.g. "§11, Open Item 3").
- Cite NASA clauses only by the IDs in square brackets, choosing ones that actually support the finding.
- Recommendations must not invent numbers, mission parameters, or approval status.
- Fill in "analysis" first: brief reasoning before you decide.
'''

def requirement_block(req_ids):
    rows = srs.set_index("req_id").loc[req_ids].reset_index()
    return "\n".join(f'{r.req_id} ({r.title}), locator "{locator(r)}": {r.text}' for r in rows.itertuples())

print(len(clause_menu), "characters of clause previews |", sorted(swe_text))

20581 characters of clause previews | ['SWE-050', 'SWE-051', 'SWE-053', 'SWE-054', 'SWE-055', 'SWE-184']


## What the LLM fills in

Smaller versions of Lauren's `Relationship`, `DocumentFinding`, and `SweCoverage`. Each has a
`case_id` so the code can match answers to benchmark cases. The code adds the rest.

In [4]:
class DraftRelationship(BaseModel):
    case_id: str
    analysis: str
    status: RelationshipStatus
    issue_code: RelationshipIssueCode
    explanation: str
    evidence: List[Quote] = Field(min_length=2, description="At least one quote from each requirement in the pair.")
    clause_ids: List[str] = Field(min_length=1)
    recommendation: str

class DraftFinding(BaseModel):
    case_id: str
    analysis: str
    criterion: DocumentCriterion
    status: DocumentStatus
    issue_code: Literal["INCOMPLETE_TRACEABILITY", "MISSING_QUALIFICATION_PROVISIONS", "MISSING_ACCEPTANCE_CRITERIA",
                        "UNNUMBERED_REQUIREMENTS", "INCOMPLETE_SAFETY_DEFINITIONS", "OTHER", "NONE"]
    searched_scope: str = Field(description="The part of the SRS you inspected, e.g. 'SRS §10'.")
    explanation: str
    evidence: List[Quote] = Field(description="May be empty only for an absence finding.")
    clause_ids: List[str] = Field(min_length=1)
    recommendation: str

class DraftSwe(BaseModel):
    case_id: str
    analysis: str
    document_evidence: DocumentEvidence
    rationale: str
    requirement_ids: List[str] = Field(description="SFMC requirement IDs whose text supports the summary, if any.")
    evidence: List[Quote]
    external_evidence_needed: List[str] = Field(description="Records outside the SRS needed to judge this SWE.")

# One answer per section holds all of that section's cases.
class RelationshipAnswer(BaseModel):
    items: List[DraftRelationship]

class FindingAnswer(BaseModel):
    items: List[DraftFinding]

class SweAnswer(BaseModel):
    items: List[DraftSwe]

## Turning drafts into report records

Each function adds what the code knows for sure, then builds Lauren's model, which runs all her checks
(quotes found in the SRS, citations among the 62 clauses, requirement IDs exist). A `ValueError` here
is sent back to the LLM.

In [5]:
def cited(clause_ids):
    ids = [cid.strip("[] ") for cid in clause_ids]  # the prompt shows IDs in brackets
    unknown = [cid for cid in ids if cid not in clauses.index]
    if unknown:
        raise ValueError(f"clause_ids that were not provided: {unknown}. Cite only the IDs in square brackets.")
    return [nasa_ref(cid) for cid in dict.fromkeys(ids)]

def check_case_ids(items, cases):
    got, expected = [d.case_id for d in items], cases.case_id.tolist()
    if sorted(got) != sorted(expected):
        raise ValueError(f"answer exactly these cases, once each: {expected} (got {got})")

def check_locators(case_id, evidence):
    bad = [q.locator for q in evidence if not q.locator.startswith("§")]
    if bad:
        raise ValueError(f"{case_id}: each locator must start with '§' (e.g. '§3.3, SFMC-REQ-015'), got {bad!r}")

def to_relationships(answer):
    check_case_ids(answer.items, rel_cases)
    pairs = dict(zip(rel_cases.case_id, rel_cases.target_ids.str.split(";")))
    out = []
    for d in answer.items:
        check_locators(d.case_id, d.evidence)
        for req_id in pairs[d.case_id]:
            if not any(req_id in q.locator for q in d.evidence):
                raise ValueError(f"{d.case_id}: needs a quote from {req_id}")
        out.append(Relationship.model_validate({
            "relationship_id": d.case_id,
            "requirement_ids": pairs[d.case_id],
            "status": d.status,
            "issue_code": d.issue_code,
            "explanation": d.explanation,
            "srs_evidence": [q.model_dump() for q in d.evidence],
            "nasa_references": cited(d.clause_ids),
            "recommendation": d.recommendation,
        }))
    return out

def to_findings(answer):
    check_case_ids(answer.items, doc_cases)
    if not any(d.criterion == "REQUIREMENT_IDENTIFIERS" for d in answer.items):
        raise ValueError("one finding must use criterion REQUIREMENT_IDENTIFIERS for the unnumbered 'shall' statements")
    for d in answer.items:
        check_locators(d.case_id, d.evidence)
    return [DocumentFinding.model_validate({
        "finding_id": d.case_id,
        "criterion": d.criterion,
        "status": d.status,
        "issue_code": d.issue_code,
        "searched_scope": d.searched_scope,
        "explanation": d.explanation,
        "srs_evidence": [q.model_dump() for q in d.evidence],
        "nasa_references": cited(d.clause_ids),
        "recommendation": d.recommendation,
    }) for d in answer.items]

def to_swe_coverage(answer):
    check_case_ids(answer.items, swe_cases)
    swe_for = dict(zip(swe_cases.case_id, swe_cases.target_ids))
    out = []
    for d in answer.items:
        swe = swe_for[d.case_id]
        check_locators(d.case_id, d.evidence)
        out.append(SweCoverage.model_validate({
            "swe_id": swe,
            "document_evidence": d.document_evidence,
            "project_compliance": "NOT_DETERMINED",  # always, per the contract
            "rationale": d.rationale,
            "supported_by_requirement_ids": list(dict.fromkeys(d.requirement_ids)),
            "srs_evidence": [q.model_dump() for q in d.evidence],
            "external_evidence_needed": list(dict.fromkeys(d.external_evidence_needed)),
            "nasa_reference": {"document": "NPR-7150.2D", "authority": "NPR_REQUIREMENT",
                               "section": f"§{CORE_SWE_SECTIONS[swe]}", "swe_id": swe, "url": NPR_CHAPTER4_URL},
        }))
    return out

## Calling the LLM

`ask()` is the same loop as 08's `review_requirement()`: use the cache if there is one, otherwise call
Gemini, check the answer, and send any errors back up to `MAX_ATTEMPTS` times.

In [6]:
def ask(name, answer_model, prompt, convert):
    cache = CACHE_DIR / f"{name}.json"
    if cache.exists():
        print(f"{name}: from cache")
        return convert(answer_model.model_validate_json(cache.read_text(encoding="utf-8")))

    from langchain.chat_models import init_chat_model
    llm = init_chat_model(f"google_genai:{MODEL}", max_retries=1).with_structured_output(answer_model, method="json_schema")
    messages = [("system", RULES), ("human", prompt)]
    for attempt in range(1, MAX_ATTEMPTS + 1):
        answer = None
        try:
            answer = with_api_retries(lambda: llm.invoke(messages))
            time.sleep(PAUSE_SECONDS)
            if answer is None:
                raise ValueError("no structured answer was returned")
            records = convert(answer)
        except ValueError as e:  # includes Pydantic's ValidationError
            print(f"{name}: attempt {attempt} failed checks")
            text = answer.model_dump_json() if isinstance(answer, answer_model) else "(unreadable answer)"
            messages += [("ai", text), ("human", f"That answer failed these checks:\n{e}\n\nFix them and answer again.")]
            continue
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(answer.model_dump_json(indent=2), encoding="utf-8")
        print(f"{name}: done in {attempt} attempt(s)")
        return records
    raise RuntimeError(f"{name}: still failing after {MAX_ATTEMPTS} attempts. Read the errors above, then run again.")

## 1. Relationships

`CONFIRMED_CONFLICT` only if both requirements can't be satisfied under the same conditions.

In [7]:
pair_text = "\n\n".join(f"{c.case_id}\n{requirement_block(c.target_ids.split(';'))}" for c in rel_cases.itertuples())

relationship_prompt = f'''Compare each pair of requirements below and decide whether they conflict.

status
- CONFIRMED_CONFLICT: both requirements cannot be satisfied under the same explicit conditions. Explain exactly why.
  Different numeric minimums are not a conflict if one value satisfies both.
- NEEDS_CLARIFICATION: a possible interaction, scope overlap, or unclear precedence, but not a proven contradiction.
- NO_CONFIRMED_CONFLICT: the requirements can both be met, even if the wording could be clearer.

issue_code: SAFETY_INTERACTION if the interaction involves a safety function or inhibit, OVERLAP_CLARIFICATION if it
is unclear which requirement governs, NONE if there is nothing to clarify, OTHER otherwise.

Give at least one quote from each requirement in the pair.

PAIRS
{pair_text}

NASA CLAUSES
{clause_menu}

FULL SRS
{srs_text}'''

relationships = ask("relationships", RelationshipAnswer, relationship_prompt, to_relationships)
for r in relationships:
    print(f"{r.relationship_id}  {' / '.join(r.requirement_ids)}  {r.status}  ({r.issue_code})\n    {r.explanation}\n")

API unavailable (GoogleAPIError), waiting 30s
API unavailable (GoogleAPIError), waiting 60s
API unavailable (GoogleAPIError), waiting 120s
relationships: done in 1 attempt(s)
REL-001  SFMC-REQ-015 / SFMC-REQ-035  NEEDS_CLARIFICATION  (SAFETY_INTERACTION)
    There is a potential scope overlap and interaction regarding whether pilot override can bypass software safety inhibits. SFMC-REQ-015 restricts weapon firing unconditionally, while SFMC-REQ-035 grants universal pilot override authority over automatic controls without defining if safety inhibits are included.

REL-002  SFMC-REQ-002 / SFMC-REQ-038  NO_CONFIRMED_CONFLICT  (NONE)
    Different numeric minimums are not a conflict because updating displayed flight information at 10 Hz (every 100 milliseconds) fulfills both the general display update constraint and the 500 millisecond position display requirement.

REL-003  SFMC-REQ-033 / SFMC-REQ-034  NO_CONFIRMED_CONFLICT  (NONE)
    The requirements work together without contradiction:

## 2. Document findings

Each case names the part of the SRS to inspect. The LLM picks the criterion; the code makes sure one
finding covers the unnumbered §2.1 "shall" statements (`REQUIREMENT_IDENTIFIERS`).

In [8]:
case_text = "\n".join(f"{c.case_id}: inspect {c.target_ids.split(':')[1]}" for c in doc_cases.itertuples())

finding_prompt = f'''Write one document-level finding for each case below. Each case names the part of the SRS to
inspect ("FULL" means the whole document).

criterion (pick the one the case is really about)
- TRACEABILITY: links between requirements and their parent needs or sources.
- QUALIFICATION_PROVISIONS: how each requirement will be verified (test, analysis, inspection, demonstration).
- ACCEPTANCE_CRITERIA: measurable pass/fail criteria for the requirements.
- REQUIREMENT_IDENTIFIERS: whether every "shall" statement has a unique requirement ID. Statements without one
  should be numbered or reworded as non-normative.
- SAFETY_DEFINITIONS: whether safety-related terms the requirements depend on are defined.
- OTHER: anything else; explain it.

status: PRESENT, PARTIAL (some content but incomplete), MISSING, or UNDETERMINED.
issue_code: the matching code, or NONE if status is PRESENT.

Evidence
- Quote the SRS text that shows what is there or what is incomplete.
- For an absence finding, evidence may be empty, but searched_scope must name exactly what you inspected.

CASES
{case_text}

NASA CLAUSES
{clause_menu}

FULL SRS
{srs_text}'''

document_findings = ask("document_findings", FindingAnswer, finding_prompt, to_findings)
for f in document_findings:
    print(f"{f.finding_id}  {f.criterion}  {f.status}  ({f.issue_code})  scope: {f.searched_scope}\n    {f.explanation}\n")

document_findings: from cache
DOC-001  TRACEABILITY  PARTIAL  (INCOMPLETE_TRACEABILITY)  scope: SRS §10
    Section 10 identifies high-level mission needs but lacks mapping between the software requirements and these parent needs.

DOC-002  QUALIFICATION_PROVISIONS  MISSING  (MISSING_QUALIFICATION_PROVISIONS)  scope: FULL
    The SRS contains no section specifying qualification provisions or verification methods for the defined software requirements.

DOC-003  ACCEPTANCE_CRITERIA  PARTIAL  (MISSING_ACCEPTANCE_CRITERIA)  scope: SRS §§3–9
    Several requirement statements rely on ambiguous, non-measurable language instead of quantitative acceptance criteria.

DOC-004  REQUIREMENT_IDENTIFIERS  PARTIAL  (UNNUMBERED_REQUIREMENTS)  scope: SRS §2.1
    Section 2.1 includes normative requirements containing 'shall' that lack requirement tags/identifiers.

DOC-005  SAFETY_DEFINITIONS  PARTIAL  (INCOMPLETE_SAFETY_DEFINITIONS)  scope: SRS §§3.3,4,11
    Key terms governing safety-critical functi

## 3. SWE coverage

One summary per §4.1 SWE. `project_compliance` is always `NOT_DETERMINED` (the code sets it), and
process rules that an SRS can't show are `NOT_ASSESSABLE`.

In [9]:
swe_list = "\n".join(f"{c.case_id}: {c.target_ids} (NPR 7150.2D §{CORE_SWE_SECTIONS[c.target_ids]}): "
                     f"{swe_text[c.target_ids]}" for c in swe_cases.itertuples())

swe_prompt = f'''For each NPR 7150.2D §4.1 provision below, summarize what evidence the SRS itself shows.

document_evidence
- DOCUMENTED: the SRS clearly contains content that this provision calls for.
- PARTIAL_EVIDENCE: some relevant content is in the SRS, but it is incomplete or other parts can't be established.
- NO_SRS_EVIDENCE: an indicator you could find in an SRS was looked for and not found. This does not mean the
  project failed to do anything.
- NOT_ASSESSABLE: the provision is about a project process (for example change control, analysis, validation,
  tracking issues to closure) that an SRS cannot show either way.

Every provision is the project manager's obligation, so the SRS can at most show evidence. List the records outside
the SRS that a reviewer would need in external_evidence_needed. Quote SRS text only where it supports the summary,
and list the requirement IDs it comes from.

PROVISIONS
{swe_list}

FULL SRS
{srs_text}'''

swe_coverage = ask("swe_coverage", SweAnswer, swe_prompt, to_swe_coverage)
for s in swe_coverage:
    print(f"{s.swe_id}  {s.document_evidence}  supported by: {', '.join(s.supported_by_requirement_ids) or '-'}\n    {s.rationale}\n")

swe_coverage: from cache
SWE-050  PARTIAL_EVIDENCE  supported by: SFMC-REQ-001
    Software requirements are established and recorded within the SRS document. However, requirements for COTS, GOTS, MOTS, OSS, or reused software components are completely absent, and formal approval/maintenance records reside outside the document.

SWE-051  PARTIAL_EVIDENCE  supported by: -
    High-level mission needs are mapped in Section 10, but the document explicitly states that a complete requirement-to-mission-need traceability matrix has not yet been developed and leaves key parameters unresolved in Section 11.

SWE-184  DOCUMENTED  supported by: SFMC-REQ-015, SFMC-REQ-035
    Software-related safety constraints and controls are explicitly documented in the SRS, including restrictions on firing at friendly spacecraft, automated threat countermeasures, and pilot command override capabilities.

SWE-053  NOT_ASSESSABLE  supported by: -
    The requirement mandates a management process for tracking an

## Assemble and validate report.json

`requirement_reviews` stays exactly as 08 wrote it. The run becomes `FULL_SRS`.

In [10]:
run = report["run"]
run["evaluation_scope"] = "FULL_SRS"
run["generated_at_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
if "notebook 09" not in run["method"]:
    run["method"] += (f"; relationships, document findings, and SWE coverage by {MODEL} in notebook 09 "
                      "(one call per section, whole SRS in context), citations filled from addressable_clauses.csv")

new_limit = "Relationships cover only the three benchmark requirement pairs; other pairs were not compared."
run["limitations"] = [l for l in run["limitations"] if not l.startswith("Requirement reviews only")]
if new_limit not in run["limitations"]:
    run["limitations"].append(new_limit)

report["relationships"] = [r.model_dump() for r in relationships]
report["document_findings"] = [f.model_dump() for f in document_findings]
report["swe_coverage"] = [s.model_dump() for s in swe_coverage]

Report.model_validate(report).to_json(REPORT_PATH)  # Lauren's checks, including the FULL_SRS rules
print("validate_report.py counts (reviews, relationships, document findings, SWEs):", validate(REPORT_PATH))

validate_report.py counts (reviews, relationships, document findings, SWEs): (54, 3, 5, 6)


## Compare with the calibration examples

The four calibration cases for these sections: REL-002, DOC-001, SWE-001, and SWE-003.
`SweCoverage` has no issue-code field, so SWE cases are compared on `document_evidence` only.

In [11]:
predicted = {r.relationship_id: (r.status, r.issue_code) for r in relationships}
predicted |= {f.finding_id: (f.status, f.issue_code) for f in document_findings}
swe_case = dict(zip(swe_cases.target_ids, swe_cases.case_id))
predicted |= {swe_case[s.swe_id]: (s.document_evidence, None) for s in swe_coverage}

cal = calibration[calibration.case_id.isin(["REL-002", "DOC-001", "SWE-001", "SWE-003"])]
cal = cal[["case_id", "expected_outcome", "issue_code"]].copy()
cal["predicted"] = cal.case_id.map(lambda c: predicted[c][0])
cal["predicted_code"] = cal.case_id.map(lambda c: predicted[c][1] or "-")
cal["status_ok"] = cal.predicted == cal.expected_outcome
print(f"status matches: {cal.status_ok.sum()}/{len(cal)}")
cal

status matches: 3/4


,case_id,expected_outcome,issue_code,predicted,predicted_code,status_ok
7,REL-002,NO_CONFIRMED_CONFLICT,NONE,NO_CONFIRMED_CONFLICT,NONE,True
8,DOC-001,PARTIAL,INCOMPLETE_TRACEABILITY,PARTIAL,INCOMPLETE_TRACEABILITY,True
9,SWE-001,PARTIAL_EVIDENCE,INCOMPLETE_LIFECYCLE_EVIDENCE,PARTIAL_EVIDENCE,-,True
10,SWE-003,PARTIAL_EVIDENCE,INCOMPLETE_SAFETY_DEFINITIONS,DOCUMENTED,-,False


### Differences explained

**Status matches: 3/4.**

- **REL-002.** Matches (`NO_CONFIRMED_CONFLICT`, `NONE`). The model saw that 10 updates per second also meets the 500 ms floor, so both can hold at once.
- **DOC-001.** Matches (`PARTIAL`, `INCOMPLETE_TRACEABILITY`). It quoted §10 saying the traceability matrix hasn't been built yet.
- **SWE-001.** Matches (`PARTIAL_EVIDENCE`). Requirements are recorded in the SRS, but approval and maintenance records live outside it.
- **SWE-003.** Doesn't match. The model said `DOCUMENTED`; expected is `PARTIAL_EVIDENCE`. It found real safety controls (SFMC-REQ-015, SFMC-REQ-035) and stopped there. It didn't consider that the SRS can't show the safety constraints are complete, and the unresolved safety open items in §11 point the other way. I'd call this a real error, not a judgment call. I didn't tune the prompt for it since it's a calibration case.

SWE cases are compared on `document_evidence` only, because `SweCoverage` has no issue-code field.

## Checks

In [12]:
saved = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
assert saved["run"]["evaluation_scope"] == "FULL_SRS"
assert len(saved["requirement_reviews"]) == 54, "requirement reviews unchanged"
assert {r["relationship_id"] for r in saved["relationships"]} == set(rel_cases.case_id)
assert {f["finding_id"] for f in saved["document_findings"]} == set(doc_cases.case_id)
assert {s["swe_id"] for s in saved["swe_coverage"]} == set(CORE_SWE_SECTIONS)
assert all(s["project_compliance"] == "NOT_DETERMINED" for s in saved["swe_coverage"])
assert any(f["criterion"] == "REQUIREMENT_IDENTIFIERS" for f in saved["document_findings"])
print("all benchmark cases answered; report.json is FULL_SRS and passes validate_report.py")

all benchmark cases answered; report.json is FULL_SRS and passes validate_report.py
